# 02 — Main Experiment: Convergent Validity & Cut-off Analysis

Two analyses that operationalise two axes of the validation framework:

- **Axis 3 (Convergent validity):** cross-rater agreement on the Dow 30
  three-rater sample (Pearson, Spearman, weighted Cohen's kappa), overall
  and by E/S/G pillar.
- **Axis 5 (Cut-off justification):** reverse-engineer the hard-coded
  grade boundaries in the broad dataset and compare them with
  distribution-based (quantile) cut-offs; quantify grade compression.

All numeric outputs are written to `results/table/` for notebook 03.


In [1]:
# ============================================================
# 02 — Main experiment
# Convergent validity (Dow 30) + cut-off analysis (broad dataset)
# Run from notebooks/.
# ============================================================
import os
import numpy as np
import pandas as pd
from scipy import stats

DATA_DIR = os.path.join("..", "data")
TAB_DIR  = os.path.join("..", "results", "table")
os.makedirs(TAB_DIR, exist_ok=True)

DOW30 = os.path.join(DATA_DIR, "Dow30-DATASET-Q2_2024.csv")
BROAD = os.path.join(DATA_DIR, "data.csv")

# ============================================================
# PART A — Convergent validity across three raters (Dow 30)
# ============================================================
dow = pd.read_csv(DOW30)

# Complete cases for the three headline raters
core = dow[["Symbol", "SNP", "Sustainalytics", "MSCI"]].dropna().copy()

# ---- Direction alignment: make ALL measures "higher = better ESG" ----
# S&P Global      : already higher = better
# Sustainalytics  : ESG RISK score, lower = better  -> sign-flip
# MSCI            : CCC..AAA -> 1..7, higher = better
msci_map = {"CCC": 1, "B": 2, "BB": 3, "BBB": 4, "A": 5, "AA": 6, "AAA": 7}
core["MSCI_num"] = core["MSCI"].map(msci_map)
core["SP_good"]   = core["SNP"]
core["SUS_good"]  = -core["Sustainalytics"]   # CRITICAL sign reversal
core["MSCI_good"] = core["MSCI_num"]

n = len(core)
print(f"Convergent-validity sample: n = {n} (Dow 30 complete cases)\n")

def pairwise_agreement(df, cols, labels):
    """Return a tidy DataFrame of Pearson/Spearman for each pair."""
    rows = []
    for (a, b), lab in zip(cols, labels):
        pr, pp = stats.pearsonr(df[a], df[b])
        sr, sp = stats.spearmanr(df[a], df[b])
        rows.append({"pair": lab, "pearson_r": pr, "pearson_p": pp,
                     "spearman_rho": sr, "spearman_p": sp})
    return pd.DataFrame(rows)

pairs  = [("SP_good", "SUS_good"), ("SP_good", "MSCI_good"),
          ("SUS_good", "MSCI_good")]
labels = ["S&P vs Sustainalytics", "S&P vs MSCI",
          "Sustainalytics vs MSCI"]

overall = pairwise_agreement(core, pairs, labels)
mean_row = pd.DataFrame([{
    "pair": "MEAN",
    "pearson_r": overall["pearson_r"].mean(), "pearson_p": np.nan,
    "spearman_rho": overall["spearman_rho"].mean(), "spearman_p": np.nan}])
overall = pd.concat([overall, mean_row], ignore_index=True)

print("Overall convergent validity (direction-aligned):")
print(overall.round(3).to_string(index=False))
print("\nBenchmarks: Berg et al. (2022) mean ~0.53-0.56; credit ratings ~0.99")

overall.to_csv(os.path.join(TAB_DIR, "convergent_validity_overall.csv"),
               index=False)

# ---- Weighted Cohen's kappa on ordinal categories ----
# IMPORTANT: forcing an ordinal grade with heavy ties (MSCI) through
# pd.qcut splits tied grades across bins and distorts the contingency
# table (it can even drive two different pairs to the same kappa). We
# therefore keep native ordinal grades intact and only quantile-bin the
# truly continuous scores (S&P, Sustainalytics).
def to_ordinal(s, n_bins=4):
    """Continuous score -> n_bins quantile categories; but if the score
    already has few distinct values (a native ordinal grade), use dense
    ranking so tied grades stay in the same category."""
    if s.nunique() <= n_bins + 2:
        return s.rank(method="dense").astype(int) - 1
    return pd.qcut(s.rank(method="first"), n_bins, labels=False)

def weighted_kappa(x, y):
    """Quadratic-weighted Cohen's kappa on two ordinal-coded vectors,
    computed over the shared union of categories."""
    a = to_ordinal(x)
    b = to_ordinal(y)
    cats = np.union1d(a.unique(), b.unique())
    k = len(cats)
    idx = {c: i for i, c in enumerate(cats)}
    ai = a.map(idx).values
    bi = b.map(idx).values
    O = np.zeros((k, k))
    for xi, yi in zip(ai, bi):
        O[xi, yi] += 1
    N = O.sum()
    row, col = O.sum(1), O.sum(0)
    E = np.outer(row, col) / N
    W = (np.subtract.outer(range(k), range(k)) ** 2) / ((k - 1) ** 2 if k > 1 else 1)
    denom = (W * E).sum()
    return 1 - (W * O).sum() / denom if denom > 0 else np.nan

kappa_rows = []
for (a, b), lab in zip(pairs, labels):
    kappa_rows.append({"pair": lab,
                       "weighted_kappa": weighted_kappa(core[a], core[b])})
kappa_df = pd.DataFrame(kappa_rows)
print("\nQuadratic-weighted Cohen's kappa (ordinal-aware):")
print(kappa_df.round(3).to_string(index=False))

# Optional: inspect contingency tables (uncomment to verify)
# for (a, b), lab in zip(pairs, labels):
#     print(f"\n[{lab}]")
#     print(pd.crosstab(to_ordinal(core[a]), to_ordinal(core[b])))

kappa_df.to_csv(os.path.join(TAB_DIR, "convergent_validity_kappa.csv"),
                index=False)

# ---- By-pillar convergent validity (S&P has E/S/G sub-scores) ----
# Only S&P exposes pillar-level scores here; we compare S&P pillars with
# the overall Sustainalytics/MSCI to illustrate pillar heterogeneity.
pillar_map = {
    "E": "SNP-environmental",
    "S": "SNP-social",
    "G": "SNP-governance",
}
pillar_rows = []
for p, colname in pillar_map.items():
    if colname in dow.columns:
        sub = dow[[colname, "Sustainalytics", "MSCI"]].dropna().copy()
        sub["MSCI_num"] = sub["MSCI"].map(msci_map)
        sub["SUS_good"] = -sub["Sustainalytics"]
        if len(sub) >= 5:
            r_sus, _ = stats.spearmanr(sub[colname], sub["SUS_good"])
            r_msci, _ = stats.spearmanr(sub[colname], sub["MSCI_num"])
            pillar_rows.append({"pillar": p, "n": len(sub),
                                "spearman_vs_Sustainalytics": r_sus,
                                "spearman_vs_MSCI": r_msci})
pillar_df = pd.DataFrame(pillar_rows)
if not pillar_df.empty:
    print("\nBy-pillar convergent validity (S&P pillar vs other raters):")
    print(pillar_df.round(3).to_string(index=False))
    pillar_df.to_csv(os.path.join(TAB_DIR, "convergent_validity_by_pillar.csv"),
                     index=False)

# ============================================================
# PART B — Cut-off analysis (broad dataset)
# ============================================================
broad = pd.read_csv(BROAD)
print("\n" + "=" * 60)
print(f"Cut-off analysis: n = {len(broad)}")
print("=" * 60)

# Reverse-engineer hard-coded boundaries
bounds = (broad.groupby("total_grade")["total_score"]
          .agg(["min", "max", "count"]).sort_values("min"))
print("\nObserved grade boundaries (hard-coded):")
print(bounds.to_string())

# What distribution-based quartile cut-offs WOULD have been
q = broad["total_score"].quantile([0.25, 0.5, 0.75])
print("\nHard-coded vs distribution-based cut-offs:")
print(f"  hard-coded (from data): ~750 / ~900 / ~1200 (round numbers)")
print(f"  quartile-based        : {q.loc[0.25]:.0f} / "
      f"{q.loc[0.5]:.0f} / {q.loc[0.75]:.0f}")

# Grade compression: share of firms in each grade
share = (broad["total_grade"].value_counts(normalize=True) * 100).round(1)
print("\nGrade share (%) — compression check:")
print(share.to_string())

cutoff_summary = pd.DataFrame({
    "metric": ["q25", "q50", "q75", "max_grade_share_pct", "n_firms"],
    "value": [q.loc[0.25], q.loc[0.5], q.loc[0.75],
              share.max(), len(broad)],
})
cutoff_summary.to_csv(os.path.join(TAB_DIR, "cutoff_summary.csv"), index=False)
bounds.to_csv(os.path.join(TAB_DIR, "cutoff_grade_boundaries.csv"))
share.rename("share_pct").to_csv(os.path.join(TAB_DIR, "cutoff_grade_share.csv"))

print("\nAll experiment tables saved to results/table/. Experiment complete.")


Convergent-validity sample: n = 26 (Dow 30 complete cases)

Overall convergent validity (direction-aligned):
                  pair  pearson_r  pearson_p  spearman_rho  spearman_p
 S&P vs Sustainalytics     -0.163      0.427         0.021       0.919
           S&P vs MSCI      0.251      0.217         0.266       0.188
Sustainalytics vs MSCI      0.285      0.158         0.295       0.144
                  MEAN      0.124        NaN         0.194         NaN

Benchmarks: Berg et al. (2022) mean ~0.53-0.56; credit ratings ~0.99

Quadratic-weighted Cohen's kappa (ordinal-aware):
                  pair  weighted_kappa
 S&P vs Sustainalytics           0.072
           S&P vs MSCI           0.211
Sustainalytics vs MSCI           0.281

By-pillar convergent validity (S&P pillar vs other raters):
pillar  n  spearman_vs_Sustainalytics  spearman_vs_MSCI
     E 25                       0.450             0.117
     S 25                       0.136             0.164
     G 25                     